# Training Setup — RunPod
Run cells top-to-bottom on a fresh RunPod instance. Each section only needs to run once per pod.

## 1. Pull latest code

In [ ]:
!git -C /workspace/Pattern_Recog_Project pull

## 2. System dependencies (Java for METEOR/CIDEr metrics)

In [ ]:
# Update package list first, then install Java
!apt-get update -q
!apt-get install -y --no-install-recommends default-jre-headless -q
!java -version

## 3. Python dependencies

In [ ]:
# Pin torch to CUDA 11.8 — prevents accidental upgrades that break CUDA
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 \
    --index-url https://download.pytorch.org/whl/cu118 -q

In [ ]:
!pip install timm pycocoevalcap pycocotools triton -q

## 4. Install mamba-ssm CUDA kernel (VMamba speedup)
Compiles from source — takes **20-40 min**. Skip if using `--encoder vit_base`.

In [ ]:
!pip install mamba-ssm --no-build-isolation

## 5. Cache pretrained weights (survives pod restarts)

In [ ]:
import os
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.makedirs("/workspace/hf_cache", exist_ok=True)
print("HF_HOME:", os.environ["HF_HOME"])

## 6. Working directory + path

In [ ]:
import sys, os
PROJECT = "/workspace/Pattern_Recog_Project"
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print("Working directory:", os.getcwd())

## 7. Download dataset

In [ ]:
!python src/data/make_data.py
!python src/data/preprocess_annotations.py

## 8. Verify setup

In [ ]:
import torch
from src.models.encoder_vmamba import WITH_MAMBA_SSM

print(f"PyTorch      : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CUDA kernel   : {WITH_MAMBA_SSM}  ← True = fast VMamba selective scan")

import subprocess, shutil
java_ok = shutil.which("java") is not None
print(f"Java (metrics): {java_ok}  ← True = METEOR/CIDEr will work")

data_ok = os.path.exists("data/flickr8k_annotations.json")
print(f"Flickr8k data : {data_ok}  ← True = dataset ready")

## 9. Config — edit before training

In [ ]:
ENCODER     = "vmamba_small"   # vit_base | vit_small | vmamba_small
DECODER     = "transformer"    # transformer | mamba | mamba3
DATASET     = "flickr8k"       # flickr8k | mscoco
BATCH_SIZE  = 16               # 64 for vit_base, 16 for vmamba_small
EPOCHS      = 30
LR          = 4e-4
WORKERS     = 8
FREEZE_ENC  = True             # freeze encoder weights, only train decoder

## 10. Train

In [ ]:
freeze_flag = "--freeze_encoder" if FREEZE_ENC else ""

!python src/models/train.py \
    --dataset {DATASET} \
    --encoder {ENCODER} \
    --decoder {DECODER} \
    {freeze_flag} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --epochs {EPOCHS} \
    --workers {WORKERS}